# Formulario: localización de dispositivos por señal WiFi

Ejecutar primero la celda de instalación, reiniciar el kernel (**Kernel → Restart**) y después ejecutar la celda del formulario.

In [ ]:
# Instalar paquete ipywidgets (solo la primera vez)
%pip install ipywidgets

In [2]:
# Instalar paquetes pandas y numpy, pip install pandas numpy
import pandas
import numpy
from sklearn import preprocessing
from sklearn import model_selection
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix


wifi = pandas.read_csv('wireless+indoor+localization/wifi_localization_discretizado.csv', header=None,
                       names=['WS1', 'WS2', 'WS3', 'WS4', 'WS5', 'WS6', 'WS7', 'Room'])

atributos = wifi.loc[:, 'WS1':'WS7']  # selección de las columnas de atributos
objetivo = wifi['Room']  # selección de la columna objetivo

# El codificador adecuado para los atributos es OrdinalEncoder, ya que permite
# trabajar con el array completo de valores de los atributos.
codificador_atributos = preprocessing.OrdinalEncoder()
codificador_atributos.fit(atributos)

# Codificación de los atributos una vez ajustado el codificador
atributos_codificados = codificador_atributos.transform(atributos)

# El codificador adecuado para la variable objetivo es LabelEncoder, que trabaja
# con la lista de sus valores, en lugar de con un array.
codificador_objetivo = preprocessing.LabelEncoder()

# El método fit_transform ajusta la codificación y la aplica a los datos a continuación
objetivo_codificado = codificador_objetivo.fit_transform(objetivo)

(atributos_entrenamiento, atributos_prueba,
 objetivo_entrenamiento, objetivo_prueba) = model_selection.train_test_split(
        # Conjuntos de datos a dividir, usando los mismos índices para ambos
        atributos_codificados, objetivo_codificado,
        # Valor de la semilla aleatoria, para que el muestreo sea reproducible,
        # a pesar de ser aleatorio
        random_state=12345,
        # Tamaño del conjunto de prueba
        test_size=.33,
        # Estratificamos respecto a la distribución de valores en la variable objetivo
        stratify=objetivo_codificado)

# En scikit-learn, ID3 se especifica mediante criterion='entropy'.
# Nota: La implementación nativa de sklearn soporta divisiones numéricas/ordinales.
modelo_id3 = DecisionTreeClassifier(
    criterion='entropy',  # Usa Ganancia de Información (Entropía) como en ID3
    random_state=12345
)

# Entrenamiento del modelo
modelo_id3.fit(atributos_entrenamiento, objetivo_entrenamiento)

# Evaluación del modelo
predicciones = modelo_id3.predict(atributos_prueba)

# Métricas de precisión general
precision_general = accuracy_score(objetivo_prueba, predicciones)




#########################
####    FORMULARIO   ####
#########################

# 1. Instalar paquete ipywidgets.
#    El paquete ipywidgets (también conocido como jupyter-widgets) 
#    es una biblioteca de Python que permite crear elementos gráficos interactivos

#    Instalar paquete IPython. 
#    Contiene metodos para usar etiquetas HTML.

import ipywidgets as widgets
from IPython.display import display, HTML

# crear variables que definen el ancho del formulario y sus controles
style = {'description_width': '130px'}
layout = widgets.Layout(width='340px')

# Cada desplegable ofrece 5 niveles de señal. Cada nivel equivale a un valor
# del fichero discretizado, elegido según los valores habituales de ese router
# (los routers 2 y 3 solo tienen 3-4 valores distintos, por eso repiten alguno).
niveles = ['Muy fuerte', 'Fuerte', 'Normal', 'Débil', 'Muy débil']

equivalencias = {
    #        Muy fuerte, Fuerte, Normal, Débil, Muy débil
    'WS1': [-2, -4, -5, -6, -7],
    'WS2': [-5, -5, -6, -7, -7],
    'WS3': [-4, -5, -6, -7, -7],
    'WS4': [-2, -4, -5, -6, -7],
    'WS5': [-4, -5, -6, -7, -8],
    'WS6': [-6, -7, -8, -9, -10],
    'WS7': [-6, -7, -8, -9, -10],
}

# Nivel por defecto de cada desplegable (una medición típica de la habitación 1)
nivel_por_defecto = {'WS1': 'Débil', 'WS2': 'Normal', 'WS3': 'Normal', 'WS4': 'Débil',
                     'WS5': 'Débil', 'WS6': 'Normal', 'WS7': 'Normal'}

# 2. Crear los elementos del formulario (selectores desplegables)
dropdowns = {}
for i, columna in enumerate(atributos.columns):
    dropdowns[columna] = widgets.Dropdown(
        # Cada opción muestra el nivel y su valor: "Fuerte (-4)"
        options=[(f"{nivel} ({equivalencias[columna][j]})", nivel)
                 for j, nivel in enumerate(niveles)],
        value=nivel_por_defecto[columna],
        description=f'{columna} (Router {i + 1}):',
        style=style,
        layout=layout
    )

# Crear los elementos del formulario (Botón del formulario para evaluar la petición)
btn_predict = widgets.Button(
    description='Localizar dispositivo',
    button_style='primary',
    icon='wifi',
    layout=widgets.Layout(width='340px', margin='15px 0 0 0')
)

# Variable contenedora que contiene contenido de los controles del formulario
out_result = widgets.Output()

# 3. Función de enlace (Front-end con Back-end)
def localizar_dispositivo(b):
    with out_result:
        out_result.clear_output()
        
        # Captura los valores del formulario HTML
        # (cada nivel elegido se convierte en su valor numérico del fichero)
        nueva_medicion = pandas.DataFrame([{
            columna: equivalencias[columna][niveles.index(dropdowns[columna].value)]
            for columna in atributos.columns
        }])
        
        # Usamos .transform() 
        nuevos_atributos_codificados = codificador_atributos.transform(nueva_medicion)

        # Realizamos la predicción numérica con el modelo ID3
        predicciones_numericas = modelo_id3.predict(nuevos_atributos_codificados)

        # Convertimos la predicción numérica (0-3) de vuelta a su habitación original (1-4)
        habitacion = codificador_objetivo.inverse_transform(predicciones_numericas)[0]
        
        # Formato visual del resultado
        colores = {1: '#3498db', 2: '#2ecc71', 3: '#f39c12', 4: '#9b59b6'}
        color = colores.get(habitacion, '#333')

        resultado = f"Habitación {habitacion}"
        
        display(HTML(f"""
            <div style="background-color: {color}; color: white; padding: 15px; border-radius: 8px; margin-top: 15px; text-align: center; font-family: sans-serif;">
                <h3 style="margin: 0; font-size: 1.1rem;">El dispositivo se encuentra en:</h3>
                <h1 style="margin: 5px 0 0 0; font-size: 2rem; text-transform: uppercase;">{resultado}</h1>
            </div>
        """))

btn_predict.on_click(localizar_dispositivo)

# 3. Mostrar el formulario en pantalla
formulario_box = widgets.VBox([
    widgets.HTML("<h3>📶 Localización de dispositivos por señal WiFi</h3>"),
    widgets.HTML(f"<p>Exactitud del modelo: <b>{precision_general * 100:.2f}%</b></p>"),
    *dropdowns.values(),
    btn_predict,
    out_result
])

display(formulario_box)